In [1]:
from stxntx.config import RAW_DIR

for sub in ["pbp", "game_details", "schedule", "overrides"]:
    (RAW_DIR / sub).mkdir(parents=True, exist_ok=True)

RAW_DIR

PosixPath('/home/arhuberg/stxntx/data/raw')

In [2]:
from nba_api.stats.endpoints import leaguegamefinder

GSW = 1610612744
games = leaguegamefinder.LeagueGameFinder(
    team_id_nullable=GSW,
    season_nullable="2016-17",
    season_type_nullable="Regular Season",
    timeout=60,
).get_data_frames()[0]

games = games.sort_values("GAME_DATE")
games[["GAME_ID", "GAME_DATE", "MATCHUP", "WL", "PTS"]].head()

,GAME_ID,GAME_DATE,MATCHUP,WL,PTS
81,0021600003,2016-10-25,GSW vs. SAS,L,100
80,0021600025,2016-10-28,GSW @ NOP,W,122
79,0021600036,2016-10-30,GSW @ PHX,W,106
78,0021600054,2016-11-01,GSW @ POR,W,127
77,0021600069,2016-11-03,GSW vs. OKC,W,122


In [3]:
from stxntx.sources import prepare_season, make_client, season_start_year

game_id = games["GAME_ID"].iloc[0]
year = season_start_year(game_id)
prepare_season(year)
client = make_client(year)
game = client.Game(game_id)

print(game_id, len(game.possessions.items), "possessions")

0021600003 201 possessions


In [4]:
p = game.possessions.items[10]
print(p)
for e in p.events:
    print("   ", e)

<Possession GameId: 0021600003, Period: 1, Number: 11, StartTime: 9:28, EndTime: 9:10, OffenseTeamId: 1610612759>
    <StatsFieldGoal GameId: 0021600003, Description: MISS Gasol 18' Jump Shot, Time: 9:11, EventNum: 17>
    <StatsRebound GameId: 0021600003, Description: Green REBOUND (Off:0 Def:1), Time: 9:10, EventNum: 18>


In [5]:
import pandas as pd

def to_seconds(clock):
    minutes, seconds = clock.split(":")
    return int(minutes) * 60 + float(seconds)

rows = []
for p in game.possessions.items:
    off = p.offense_team_id
    start_score = p.previous_possession.events[-1].score if p.previous_possession else p.events[0].score
    end_score = p.events[-1].score
    rows.append({
        "period": p.period,
        "number": p.number,
        "offense_team_id": off,
        "start_clock": p.start_time,
        "end_clock": p.end_time,
        "start_seconds": to_seconds(p.start_time),
        "end_seconds": to_seconds(p.end_time),
        "start_type": p.possession_start_type,
        "points": end_score.get(off, 0) - start_score.get(off, 0),
    })

df = pd.DataFrame(rows)
df.head(10)

,period,number,offense_team_id,start_clock,end_clock,start_seconds,end_seconds,start_type,points
0,1,1,1610612759,12:00,11:36,720.0,696.0,OffDeadball,0
1,1,2,1610612744,11:36,11:21,696.0,681.0,OffLongMidRangeMiss,0
2,1,3,1610612759,11:21,11:11,681.0,671.0,OffArc3Miss,2
3,1,4,1610612744,11:11,10:55,671.0,655.0,OffAtRimMake,2
4,1,5,1610612759,10:55,10:26,655.0,626.0,OffLongMidRangeMake,2
5,1,6,1610612744,10:26,10:19,626.0,619.0,OffShortMidRangeMake,0
6,1,7,1610612759,10:19,10:06,619.0,606.0,OffAtRimMiss,2
7,1,8,1610612744,10:06,9:52,606.0,592.0,OffLongMidRangeMake,0
8,1,9,1610612759,9:52,9:41,592.0,581.0,OffLiveBallTurnover,0
9,1,10,1610612744,9:41,9:28,581.0,568.0,OffArc3Miss,0


In [6]:
team_points = df.groupby("offense_team_id")["points"].sum()
print(team_points)
print(games.loc[games["GAME_ID"] == game_id, ["MATCHUP", "PTS"]])

offense_team_id
1610612744    100
1610612759    129
Name: points, dtype: int64
        MATCHUP  PTS
81  GSW vs. SAS  100


In [7]:
eoq = df[(df["period"] <= 3) & (df["start_seconds"] <= 50)]
eoq[["period", "number", "offense_team_id", "start_clock", "end_clock", "start_type", "points"]]

,period,number,offense_team_id,start_clock,end_clock,start_type,points
47,1,48,1610612744,0:41,0:30,OffAtRimBlock,0
48,1,49,1610612759,0:30,0:08,OffArc3Miss,3
49,1,50,1610612744,0:08,0:06,OffArc3Make,0
50,1,51,1610612759,0:06,0:02,OffLiveBallTurnover,3
51,1,52,1610612744,0:02,0:00,OffArc3Make,0
99,2,48,1610612759,0:46,0:41,OffAtRimMiss,2
100,2,49,1610612744,0:41,0:32,OffTimeout,0
101,2,50,1610612759,0:32,0:19,OffDeadball,2
102,2,51,1610612744,0:19,0:06,OffShortMidRangeMake,2
103,2,52,1610612759,0:06,0:00,OffUnknownDistance2ptMake,3
